# Data Packing Strategies
### Advanced Fine-Tuning Paradigms  ·  Colab T4 (16 GB) ready

> The previous three notebooks each left a number on the table. **CPT** packed raw text and reached ~100 % token utilisation. **Instruction Tuning** measured its own supervision-per-forwarded-token at ~25–40 % and named this concept as the fix. **MTFT** showed that mixed-length streams silently distort a loss.
> This notebook is the payoff: **three packing strategies implemented and measured side by side** — and, more importantly, the part most tutorials omit entirely. **Packing is not free.** Done naively it lets one training example attend to another and assigns wrong positional indices, which quietly corrupts the loss. Section 3 measures that corruption in loss units, then fixes it.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **Sequence packing** (a.k.a. **example packing**, **multipack**) fills a fixed context window of length `L` with **multiple independent training sequences** instead of one sequence plus padding, so that the fraction of forwarded tokens carrying real signal — **token utilisation** `U` — approaches 1:
  $$U = \frac{\sum_{i=1}^{N}\ell_i}{n_{\text{batches}} \cdot B \cdot L} \qquad\text{(padding: } U \ll 1;\ \text{packing: } U \to 1)$$
- There are exactly **two** strategies in production use, and they differ in one crucial property:
  - **Wrapped packing** (*concat-and-chunk*, the pretraining default used by GPT-2/3 and Llama): concatenate the whole tokenised corpus into one stream separated by `EOS`, then slice fixed `L`-token blocks. **Utilisation ≈ 100 %**, but documents are **split across block boundaries** — a sequence's tail begins the next block with no context.
  - **Best-Fit-Decreasing (BFD) bin packing** (*multipack*, what TRL's `packing_strategy="bfd"` and Axolotl implement): sort sequences by length descending and place each into the **fullest bin that still fits**. **No sequence is ever split**, at the cost of a small residual gap per bin.
- **The bin-packing problem is NP-hard**, so BFD is a heuristic — but a well-characterised one: **First-Fit-Decreasing uses at most `11/9·OPT + 6/9` bins** (*Dósa & Sgall, 2013*, a tight bound). In practice on real SFT length distributions it lands within a few percent of optimal, which is why nobody solves it exactly.
- **The correctness problem packing creates.** A packed block is *one* sequence as far as the model is concerned, so by default:
  1. **Attention leaks across sequence boundaries** — tokens of example 2 attend to example 1. The causal mask prevents looking *forward*, not looking at *a different document*.
  2. **Positional indices are wrong** — example 2 starts at RoPE position `ℓ₁` instead of `0`, so every sequence after the first is evaluated at positions it will never see at inference.
- **The fix has two independent halves**, and Section 3 measures each separately:
  - **`position_ids` reset to 0 at every sequence boundary.**
  - **A block-diagonal (a.k.a. *segment-wise causal*) attention mask** — implemented as `cu_seqlens` for **FlashAttention-2 varlen**, a `BlockMask` for **FlexAttention**, or an explicit **4D additive mask** for `sdpa`/`eager`. Only the first two are also *faster*; the 4D dense mask buys correctness alone (see [Context Block]).
- **Terminology worth keeping straight:** *padding waste* (pad tokens forwarded), *token utilisation* (the ratio above), *cross-contamination* (attention across sequence boundaries), *varlen / unpadded attention* (kernels that consume `cu_seqlens` and skip the masked blocks entirely).

### One-sentence definition of the mechanics

> **Data packing concatenates multiple independent training sequences into each fixed-length context window — by streaming concatenation or by bin packing — driving token utilisation from as low as 10 % to ~100 %, and requires a block-diagonal attention mask plus per-sequence `position_ids` to keep the packed batch mathematically equivalent to the unpacked one.**

### The exact engineering problem it solves

- **Padding is pure, billable waste.** A pad token consumes exactly the same FLOPs, memory bandwidth and wall-clock as a real token, and contributes exactly zero gradient. On `tatsu-lab/alpaca` (mean ≈ 80 tokens) fed into a 1024-token window one example at a time, **~92 % of every forward pass computes nothing.** That is not an inefficiency, it is a 12× bill.
- **The waste scales *up* with batch size, which is the counter-intuitive part.** Dynamic padding pads to the longest sequence *in the batch*, and the expected batch maximum grows with `B`. Bigger batches — the thing you reach for to go faster — make padding *worse*, not better.
- **Length variance is the real enemy, not length.** A corpus of uniformly 500-token sequences wastes nothing. A corpus mixing 40-token chat turns with 900-token document-QA — i.e. **every realistic instruction mixture** — wastes most of its compute, which is precisely why this concept belongs in the same chapter as MTFT.
- **It converts padding waste into *optimizer steps*.** At fixed wall-clock, packing does not merely run faster; it puts 2–10× more real tokens per gradient update, which changes the effective batch size and therefore the learning-rate regime you should be using.
- **And what packing does *not* solve:** it does not reduce attention's `O(L²)` cost — packing short sequences into long blocks *increases* attention FLOPs unless you use a varlen kernel that skips the off-diagonal blocks. Section 3 computes the exact crossover length for this model.

---

### The Human Element — Hugging Face datasets for packing

| HF path | What it is | Why it's the right corpus for this concept |
|---|---|---|
| **`databricks/databricks-dolly-15k`** (15,011 rows) | Human-written instructions across 8 task categories; **`context`** is a separate field, present on some rows and empty on others. | **This notebook's working set, chosen for its length *variance*.** `open_qa` rows are ~40–80 tokens; `closed_qa`/`information_extraction` rows carry a Wikipedia passage in `context` and run to 800+. That bimodality is exactly the distribution where the three strategies produce *different* numbers — on a uniform-length corpus they would all tie, and the notebook would prove nothing. The optional `context` field is what creates the bimodality, so it is a feature here, not noise. |
| **`tatsu-lab/alpaca`** (52,002 rows) | The canonical self-instruct SFT set: `instruction` / `input` / `output`, mean length ≈ 80 tokens. | **The pathological case, and the reason this concept exists.** Almost every row is far shorter than any sane context window, so one-example-per-sequence at `L=1024` throws away **~92 %** of the compute. It is also the historical reason packing became standard practice in open SFT — Alpaca-era recipes were burning an order of magnitude more GPU time than they needed to. |
| **`allenai/tulu-3-sft-mixture`** (939,343 rows) | The production-scale blend: chat, math, code, safety, precise instruction-following — every row tagged with `source`. | **Where BFD's advantage over wrapped packing is largest.** Its length distribution is extreme (one-line chat next to multi-thousand-token code and CoT traces), so wrapped packing would split a large fraction of the long sequences mid-token-stream while BFD keeps them intact. This is the corpus shape that makes "which packing strategy" a real decision rather than a tie. |
| **`HuggingFaceFW/fineweb-edu`** *(contrast case)* | Raw web/educational text, arbitrarily long documents. | **The case where wrapped packing is correct and BFD is pointless.** For CPT-style raw-text training the documents are *longer* than the block, so there is no bin-packing problem to solve and splitting is semantically harmless — you are modelling `p(x)` over a stream. This is why the CPT notebook used wrapped packing and this notebook does not simply repeat it: **the right strategy is a property of your length distribution.** |

**Why the length histogram is the first thing to compute, always:** every number in this notebook — utilisation, batch count, the choice between wrapped and BFD, even whether packing is worth implementing — is a deterministic function of your corpus's length distribution and your `L`. Section 3 therefore starts by plotting it in text, before any GPU work.

> This notebook packs **`databricks/databricks-dolly-15k`** into `L = 1024` blocks three ways, measures utilisation and wall-clock for each, then measures — in loss units — what naive packing does to correctness and what the two fixes recover.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **A pad token is indistinguishable from a real token to the hardware.** The GEMMs are shaped `(B·L, d)`; nothing in the kernel knows some rows are meaningless. `attention_mask=0` removes a pad token's *influence*, not its *cost*. So padding waste converts one-to-one into wasted FLOPs, wasted memory bandwidth and wasted dollars.
- **Why dynamic padding is not the answer, and why bigger batches make it worse.** Dynamic padding pads to `max(batch)`, so the waste per batch is `B·max(ℓ) − Σℓ`. For lengths drawn from any spread distribution, `E[max]` **increases monotonically with `B`** — the more you batch, the further the batch maximum sits from the batch mean. Length-grouped sampling (`group_by_length=True`) attacks this by putting similar lengths together and recovers most of it; **packing removes the failure mode entirely** by making every batch element exactly `L` real tokens.
- **Why the causal mask does not save you.** A causal mask enforces "no token attends to the future". Two independently sampled documents concatenated in one window are *both in each other's past*, so document 2's tokens legitimately (as far as the mask is concerned) attend to document 1. The model gets to condition on text that will never precede it at inference. This is **train/inference skew injected by a data-loading optimisation** — the worst kind, because throughput improves and nothing errors.
- **Why `position_ids` matter independently of the mask.** RoPE encodes *absolute* position into the rotation applied to Q and K. Without a reset, the third sequence in a block might live at positions 600–700 — a region the model will never occupy at inference for that content. Resetting per sequence is a one-line change with no compute cost, and Section 3 shows it accounts for a **measurable** share of the total packing error.
- **Why the fix is *not* automatically a speed-up — the part that gets glossed over.** There are two ways to make attention block-diagonal:
  - **Dense 4D additive mask** (`sdpa`, `eager`): you still compute the full `L×L` score matrix and then throw most of it away. **Correct, not faster.** This is the only route on a T4, and it is what this notebook implements.
  - **Varlen / unpadded kernels** (`flash_attention_2` with `cu_seqlens`, or FlexAttention's `BlockMask`): the kernel never *materialises* the off-diagonal blocks. **Correct *and* faster** — this is what production packing uses, and it needs Ampere or newer.
- **Why packing can *increase* attention FLOPs.** Attention is `O(L²)`; the FFN and projections are `O(L)`. Packing `N` sequences of length `ℓ` into one block of `L = N·ℓ` leaves the linear cost unchanged but multiplies the attention-score cost by `N`. Whether that matters depends entirely on where `L` sits relative to the **crossover length** at which attention FLOPs equal linear FLOPs — for this model it is **well over ten thousand tokens**, so at `L = 1024` attention is only a few percent of the total and packing is a large net win. Section 3 computes the crossover from this model's own config rather than asserting it.
- **Why the masks are built in the collator, not stored in the dataset.** A block-diagonal mask for `L = 1024` is `1024² = 1.05 M` entries **per example**. Materialising that in Arrow would dwarf the dataset by orders of magnitude; it is reconstructed from a compact per-token `doc_ids` vector at collate time in three tensor ops.

#### VRAM & Compute Impact

| Strategy | Token utilisation | Batches for the same corpus | Sequence splitting | Extra memory |
|---|---|---|---|---|
| **Static pad to `L`** | worst — often **8–15 %** | most | none | none |
| **Dynamic pad (random order)** | poor, **degrades as `B` grows** | many | none | none |
| **Dynamic pad + `group_by_length`** | decent | fewer | none | none |
| **Wrapped (concat-and-chunk)** | **~100 %** | fewest | **yes — every boundary** | none |
| **BFD bin packing** | **~95–99 %** | near-fewest | **never** | `doc_ids` vector |
| *+ block-diagonal 4D mask* | unchanged | unchanged | — | `B·L²` per batch |

- **The 4D mask's memory is small but not nothing:** `B=2, L=1024` in fp32 is `2 · 1024² · 4 B ≈ 8 MB`; at `B=8, L=4096` it is `537 MB`, and it grows as `L²`. Past a couple of thousand tokens you need a varlen kernel for memory reasons alone, not just speed.
- **Fully-padded rows are a NaN trap.** In a bin with leftover space, the pad positions belong to no document, so a strictly block-diagonal mask masks their entire row → `softmax(all −inf)` → **NaN that propagates through the whole batch**. The fix is to always allow the diagonal, so a pad position attends to itself and contributes nothing (its label is `-100`). This notebook does that explicitly, because the failure mode is silent until the loss becomes NaN.
- **Packing changes your effective batch size**, and therefore your learning rate. Going from ~30 % to ~98 % utilisation at fixed `B` and `L` roughly **triples the real tokens per optimizer step**. Keeping the LR unchanged is a decision, not a default.
- **T4 budget for this notebook:** three ~1–2 minute training runs, one loss-ablation cell of ~13 forward passes, and no long training at all — the deliverable is a set of measurements plus reusable packing functions. **Total ~8–12 min**, peak ~3 GB.

#### Pros & Cons

**Pros**
- **The single highest-leverage data-pipeline optimisation in fine-tuning** — 2–10× more real tokens per unit of compute on realistic instruction mixtures, for maybe 40 lines of code.
- **Strategy-appropriate correctness is achievable**: BFD never splits a sequence; a block-diagonal mask makes the packed batch mathematically equivalent to the unpacked one.
- **Composes with everything else** — CPT, IT and MTFT all get faster with no change to their objectives.
- **Fewer, larger optimizer steps** means less per-step overhead (dataloader, optimizer, all-reduce in the distributed case).
- **Widely supported now**: `SFTConfig(packing=True, packing_strategy="bfd")` in TRL, multipack in Axolotl, `cu_seqlens` in FlashAttention-2, `BlockMask` in FlexAttention.

**Cons**
- **Naive packing silently corrupts training.** No error, no warning, better throughput — and cross-document attention plus wrong positions. This is the main reason the concept deserves a full notebook rather than a flag.
- **The dense-mask fix is not a speed-up** and its memory grows as `L²`; the fix that *is* a speed-up requires Ampere+ hardware.
- **Wrapped packing splits sequences**, which for SFT means training on completion fragments with no preceding instruction. Acceptable for raw-text CPT, questionable for instruction data.
- **Bin packing is NP-hard**; you are always running a heuristic, and it needs the full length distribution up front (awkward for pure streaming pipelines).
- **Loss normalisation gets subtle.** A packed block may hold 1 long sequence or 12 short ones, so token-mean loss weights blocks unevenly — the same trap the MTFT notebook measured, now baked into the batch layout.
- **Debugging is harder.** "Which example produced this loss spike?" needs `doc_ids` bookkeeping that unpacked training gives you for free.
- **More moving parts to get wrong**: masks, positions, label alignment, and per-strategy collators all have to agree.

#### Metrics to watch

- **Token utilisation** `Σℓ / (n_batches·B·L)` — the headline number, computed *before* writing any GPU code.
- **Real tokens per optimizer step** — what actually changed about your training, and the reason to revisit the LR.
- **Wall-clock per epoch and tokens/sec** — the thing you are buying.
- **Packing error in loss units** — per-sequence loss under packing vs. the same sequences unpacked. Should be ≈ 0 with both fixes on; anything else is corruption you are training on.
- **Sequences split** (wrapped only) and **residual gap per bin** (BFD only).
- **Attention share of total FLOPs at your `L`** — tells you whether you need a varlen kernel or a dense mask is fine.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**Three packing strategies + a block-diagonal 4D attention mask + per-sequence `position_ids`, all measured against an unpacked reference.**

> ⚙️ **Why hand-written and not `SFTConfig(packing=True)`:** TRL will do this for you in one flag, and Step 8 shows exactly which flag. But the flag hides the two things worth understanding — that naive packing is *wrong*, and that the fix has two separable halves — and it cannot run the experiment that proves it. The functions here are ~40 lines and are the same algorithms TRL and Axolotl implement.

**Executable pipeline:**

| Step | What | The point |
|---|---|---|
| 1 | Model, tokenizer, knobs | `sdpa` — a T4 cannot use FlashAttention-2 varlen |
| 2 | Dolly → completion-masked examples, **length histogram** | length *variance* is what makes packing matter |
| 3 | **Analytical** comparison of 5 padding/packing schemes | pure arithmetic, no GPU, before any code |
| 4 | Strategy B: **wrapped** concat-and-chunk (masks preserved) | ~100 % utilisation, splits sequences |
| 5 | Strategy C: **BFD bin packing** | never splits, near-100 % utilisation |
| 6 | `doc_ids` → **block-diagonal 4D mask** + reset `position_ids` | the correctness machinery |
| 7 | **The ablation:** per-sequence loss, unpacked vs 4 packed variants | measures corruption *and* the fix |
| 8 | FLOPs crossover · three timed training runs · production wiring | what you actually buy |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers datasets peft accelerate bitsandbytes

In [ ]:
import os, gc, math, time, statistics
from collections import Counter

import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
# A T4 is Turing (sm_75): no bf16 tensor cores AND no FlashAttention-2, so the varlen
# (cu_seqlens) route to block-diagonal attention is unavailable. We use a dense 4D mask on
# sdpa instead: correct, but not faster — see [Context Block].
HAS_FA2 = major >= 8
print(f"sm_{major}{_minor} | bf16: {USE_BF16} | FlashAttention-2 varlen available: {HAS_FA2}")

# ---- Packing knobs ----------------------------------------------------------------
BLOCK_SIZE = 1024  # L: the context window we are trying to fill
N_EXAMPLES = 600  # corpus size for the comparison (kept small so 3 training runs are quick)
BATCH_SIZE = 4  # per-device batch; the analytical table also sweeps this

### Step 1 — Model & tokenizer

`Qwen/Qwen2.5-0.5B-Instruct`, for two reasons specific to this notebook:

- It has a **working chat template and a correct `eos_token`** (`<|im_end|>`), so each packed sequence already **self-terminates** — which is why the BFD packer below needs no extra separator token. In raw-text wrapped packing (the CPT notebook) you must insert `EOS` yourself; with a chat template it is already there. Knowing which case you are in is the difference between a correct packer and a subtly broken one.
- The **loss ablation in Step 7 needs a model whose loss is meaningful on instruction data**, so that "packing error" is measured against a sensible reference rather than noise.

`attn_implementation="sdpa"` is not a default here, it is a **constraint**: FlashAttention-2's varlen path — the one that makes block-diagonal attention *faster* rather than merely correct — requires Ampere or newer.

In [ ]:
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print(f"eos={tokenizer.eos_token!r}({tokenizer.eos_token_id}) pad={tokenizer.pad_token!r}({tokenizer.pad_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",   # a T4 cannot use flash_attention_2's varlen packing path
)
model.config.use_cache = False
print(f"{model.get_memory_footprint()/1e9:.2f} GB base (4-bit) | hidden {model.config.hidden_size} | "
      f"layers {model.config.num_hidden_layers}")

### Step 2 — The corpus, and the histogram that decides everything

Same completion-masked tokenization as the instruction-tuning notebook (render to text, tokenize the halves separately so the `-100` boundary is exact by construction). One addition matters for packing: **the label mask travels with the token ids**, because a packer that concatenates `input_ids` and forgets to concatenate `labels` in lockstep produces a dataset that trains on prompts.

The length distribution printed below is the input to every decision in this notebook. Dolly is deliberately chosen for its **bimodality** — short `open_qa` rows next to `closed_qa` rows carrying a whole passage in `context`.

In [ ]:
dolly = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=42)

def to_messages(ex):
    user = ex["instruction"] if not ex["context"] else f"{ex['instruction']}\n\n{ex['context']}"
    return [{"content": user, "role": "user"},
            {"content": ex["response"], "role": "assistant"}]

def build_example(messages):
    """Completion-masked example. Labels travel WITH input_ids — packers must move both."""
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    if not full_text.startswith(prompt_text):
        raise ValueError("chat template is not prefix-consistent")
    p = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    c = tokenizer(full_text[len(prompt_text):], add_special_tokens=False)["input_ids"]
    if len(p) + len(c) > BLOCK_SIZE:
        return None   # cannot pack what does not fit in one block at all
    return {"input_ids": p + c, "labels": [-100] * len(p) + c, "category": None}

rows, dropped = [], 0
for ex in dolly.select(range(int(N_EXAMPLES * 1.3))):
    out = build_example(to_messages(ex))
    if out is None:
        dropped += 1
    else:
        out["category"] = ex["category"]
        rows.append(out)
    if len(rows) >= N_EXAMPLES:
        break

lens = [len(r["input_ids"]) for r in rows]
print(f"examples: {len(rows)} (dropped {dropped} longer than BLOCK_SIZE={BLOCK_SIZE})")
print(f"length  : min {min(lens)} | median {statistics.median(lens):.0f} | mean {statistics.mean(lens):.0f} "
      f"| p90 {sorted(lens)[int(.9*len(lens))]} | max {max(lens)}")
print(f"stdev   : {statistics.stdev(lens):.0f}  <- VARIANCE is what makes packing pay off")

# Text histogram: the shape of this is why the strategies below differ at all.
buckets = Counter(min(int(l // 100) * 100, 1000) for l in lens)
for b in sorted(buckets):
    print(f"  {b:>4}-{b+99:<4} {'#' * max(1, round(40 * buckets[b] / max(buckets.values()))):<41} {buckets[b]}")

### Step 3 — Analytical comparison of five schemes (no GPU required)

**Do this before writing a packer.** Utilisation is a deterministic function of the length distribution and `L`, so you can decide whether packing is worth implementing — and which strategy — with pure arithmetic:

- **Static pad to `L`** — every example padded to the full window. Still common in tutorial code, and usually catastrophic.
- **Dynamic pad, random order** — pad to the batch maximum. Note how it **gets worse as `B` grows**: `E[max]` moves away from `E[ℓ]`.
- **Dynamic pad + `group_by_length`** — sort by length so batches are homogeneous. This is the *honest* baseline; it recovers most of the loss and is one flag in `TrainingArguments`.
- **Wrapped** — `⌈Σℓ / L⌉` blocks. The floor.
- **BFD** — the number of bins the packer in Step 5 actually produces.

In [ ]:
def tokens_forwarded_padded(lengths, batch_size, sort_by_length=False):
    """Dynamic padding: every batch costs max(batch) * len(batch) tokens."""
    order = sorted(lengths, reverse=True) if sort_by_length else list(lengths)
    return sum(max(order[i:i + batch_size]) * len(order[i:i + batch_size])
               for i in range(0, len(order), batch_size))

real = sum(lens)
print(f"real (non-pad) tokens in the corpus: {real:,}\n")
print(f"{'scheme':<34}{'fwd tokens':>12}{'utilisation':>13}{'waste':>9}")
print("-" * 68)

schemes = [("static pad to L", len(lens) * BLOCK_SIZE)]
for bs in (1, 2, 8, 16):
    schemes.append((f"dynamic pad, random, B={bs}", tokens_forwarded_padded(lens, bs)))
schemes.append((f"dynamic pad, group_by_length, B=8", tokens_forwarded_padded(lens, 8, sort_by_length=True)))
schemes.append(("wrapped (theoretical floor)", math.ceil(real / BLOCK_SIZE) * BLOCK_SIZE))

for name, fwd in schemes:
    print(f"{name:<34}{fwd:>12,}{100*real/fwd:>12.1f}%{100*(1-real/fwd):>8.1f}%")

print(f"\nRead the B=1..16 rows in order: dynamic padding gets WORSE as the batch grows, because")
print(f"the expected batch maximum moves away from the mean. Batching harder is not the fix.")
print(f"group_by_length recovers most of it — the honest baseline packing must beat.")

### Step 4 — Strategy B: **wrapped** packing (concat-and-chunk)

The pretraining default, and what the CPT notebook used. Concatenate everything, slice at `L`.

- **`labels` are concatenated in lockstep with `input_ids`** — this is the detail that separates wrapped packing for SFT from wrapped packing for raw text. Forget it and you train on prompts.
- **No separator is inserted**, because each sequence already ends with `<|im_end|>` from the chat template. For raw text (no template) you must append `EOS` yourself.
- **`doc_ids` records which sequence each token came from**, which is what makes the block-diagonal mask constructible later. A sequence split across two blocks appears as **two segments with different ids** — the mask can fix contamination, but *nothing* can fix the fact that the second fragment lost its instruction.
- The ragged tail is discarded, and `position_ids` restart at every segment boundary.

In [ ]:
def segment_positions(doc_ids):
    """position_ids restarting at 0 for every segment — RoPE encodes ABSOLUTE position."""
    pos, counter, prev = [], 0, None
    for d in doc_ids:
        counter = 0 if d != prev else counter + 1
        pos.append(counter)
        prev = d
    return pos

def pack_wrapped(rows, L):
    """Concat-and-chunk. ~100% utilisation; SPLITS sequences across block boundaries."""
    ids, labs, docs = [], [], []
    for d, r in enumerate(rows):
        ids.extend(r["input_ids"])
        labs.extend(r["labels"])
        docs.extend([d] * len(r["input_ids"]))   # provenance, for the block-diagonal mask

    n_blocks = len(ids) // L    # drop the ragged tail
    out, split_count = [], 0
    for b in range(n_blocks):
        s = slice(b * L, (b + 1) * L)
        block_docs = docs[s]
        # Renumber to LOCAL segment ids: a sequence split across two blocks becomes a fresh
        # segment in each, which is exactly how the mask must treat it.
        seen, local = {}, []
        for d in block_docs:
            if d not in seen:
                seen[d] = len(seen)
            local.append(seen[d])
        out.append({"input_ids": ids[s], "labels": labs[s], "doc_ids": local,
                    "position_ids": segment_positions(local)})
        # A block whose first doc also appeared in the previous block => that doc was split.
        if b > 0 and block_docs[0] == docs[b * L - 1]:
            split_count += 1

    kept, corpus = n_blocks * L, len(ids)
    print(f"wrapped: {len(out)} blocks x {L} | utilisation 100.0% (zero padding by construction) "
          f"| ragged tail DISCARDED: {corpus - kept:,} tokens ({100*(corpus-kept)/corpus:.2f}% of corpus) "
          f"| sequences split across a boundary: {split_count}")
    return out

wrapped_blocks = pack_wrapped(rows, BLOCK_SIZE)

# Self-check: no token lost or invented, labels still aligned, positions restart per segment.
_b = next(b for b in wrapped_blocks if max(b["doc_ids"]) >= 1)   # a block with >1 segment
assert len(_b["input_ids"]) == len(_b["labels"]) == len(_b["doc_ids"]) == len(_b["position_ids"]) == BLOCK_SIZE
assert all(l == t for t, l in zip(_b["input_ids"], _b["labels"]) if l != -100), "label misalignment"
assert _b["position_ids"][0] == 0
for i in range(1, BLOCK_SIZE):   # positions must reset exactly at segment changes
    assert _b["position_ids"][i] == (0 if _b["doc_ids"][i] != _b["doc_ids"][i-1]
                                     else _b["position_ids"][i-1] + 1)
print("wrapped self-check passed")

### Step 5 — Strategy C: **Best-Fit-Decreasing** bin packing

The SFT-appropriate strategy: **no sequence is ever split.**

- Sort by length **descending**, then place each sequence into the **fullest bin that still fits it** (best-fit). Longest-first is what makes the greedy choice good — it places the awkward items while the bins are still empty, then backfills gaps with short sequences.
- **Bin packing is NP-hard**; this is the standard heuristic, and FFD is provably within `11/9·OPT + 6/9` bins (*Dósa & Sgall, 2013*). Nobody solves it exactly, because a few percent of residual gap is not worth an ILP.
- Leftover space is padded and labelled `-100`, so the residual gap is real but small — the printout reports it.
- Same `doc_ids` / `position_ids` contract as wrapped, so **one collator serves both strategies**.

In [ ]:
def pack_bfd(rows, L, verbose=True):
    """Best-Fit-Decreasing bin packing: near-100% utilisation, NEVER splits a sequence.

    ponytail: O(n_items * n_bins) linear scan for the best-fit bin. Fine to ~1e4 sequences
    (this is 600). Upgrade path for a real corpus: a heap/segment tree keyed on remaining
    capacity, which is what Axolotl's multipack does.
    """
    order = sorted(range(len(rows)), key=lambda i: -len(rows[i]["input_ids"]))
    bins = []    # each: {"items": [...], "used": int}
    for i in order:
        n = len(rows[i]["input_ids"])
        best, best_rem = None, None
        for b in bins:
            rem = L - b["used"] - n
            if rem >= 0 and (best_rem is None or rem < best_rem):   # BEST fit = smallest gap left
                best, best_rem = b, rem
        if best is None:
            bins.append({"items": [i], "used": n})
        else:
            best["items"].append(i)
            best["used"] += n

    out = []
    for b in bins:
        ids, labs, docs = [], [], []
        for seg, i in enumerate(b["items"]):
            ids.extend(rows[i]["input_ids"])
            labs.extend(rows[i]["labels"])
            docs.extend([seg] * len(rows[i]["input_ids"]))
        pad = L - len(ids)
        out.append({
            "input_ids": ids + [tokenizer.pad_token_id] * pad,
            "labels": labs + [-100] * pad,    # padding must never be supervised
            "doc_ids": docs + [-1] * pad,    # -1 = belongs to NO sequence
            "position_ids": segment_positions(docs) + [0] * pad,
        })

    if verbose:
        used = sum(b["used"] for b in bins)
        per_bin = [b["used"] for b in bins]
        print(f"BFD: {len(bins)} bins x {L} | utilisation {100*used/(len(bins)*L):.1f}% "
              f"| sequences per bin: min {min(len(b['items']) for b in bins)}, "
              f"max {max(len(b['items']) for b in bins)} "
              f"| residual gap: mean {L - statistics.mean(per_bin):.0f} tok")
    return out

bfd_blocks = pack_bfd(rows, BLOCK_SIZE)

# Self-check: every sequence placed exactly once, nothing split, padding never supervised.
_placed = sum(sum(1 for d in set(b["doc_ids"]) if d >= 0) for b in bfd_blocks)
assert _placed == len(rows), f"placed {_placed} sequences but had {len(rows)}"
_b = bfd_blocks[0]
assert len(_b["input_ids"]) == BLOCK_SIZE and len(_b["doc_ids"]) == BLOCK_SIZE
assert all(l == -100 for d, l in zip(_b["doc_ids"], _b["labels"]) if d < 0), "pad token supervised"
_runs = [d for i, d in enumerate(_b["doc_ids"]) if i == 0 or d != _b["doc_ids"][i-1]]
assert len(_runs) == len(set(_runs)), "a sequence appears in two runs => it was split"
print("BFD self-check passed")

# `real_seen` differs per strategy: wrapped DROPS its ragged tail, so it does not process the
# whole corpus. Dividing the full corpus by its forwarded tokens is how you get a nonsense "utilisation > 100%".
wrapped_fwd, bfd_fwd = len(wrapped_blocks) * BLOCK_SIZE, len(bfd_blocks) * BLOCK_SIZE
table = [
    ("dynamic pad (random)", tokens_forwarded_padded(lens, BATCH_SIZE), sum(lens), math.ceil(len(lens)/BATCH_SIZE)),
    ("dynamic pad (grouped)", tokens_forwarded_padded(lens, BATCH_SIZE, sort_by_length=True), sum(lens), math.ceil(len(lens)/BATCH_SIZE)),
    ("wrapped", wrapped_fwd, wrapped_fwd, math.ceil(len(wrapped_blocks)/BATCH_SIZE)),
    ("BFD", bfd_fwd, sum(lens), math.ceil(len(bfd_blocks)/BATCH_SIZE)),
]
print(f"\n{'strategy':<24}{'batches @B=' + str(BATCH_SIZE):>15}{'fwd tokens':>13}{'real seen':>12}{'util':>8}")
print("-" * 72)
for name, fwd, seen, nb in table:
    print(f"{name:<24}{nb:>15}{fwd:>13,}{seen:>12,}{100*seen/fwd:>7.1f}%")
print(f"\nreal tokens per optimizer-step-worth of batch is the number that changed most:")
print(f"  grouped padding: ~{tokens_forwarded_padded(lens, BATCH_SIZE, sort_by_length=True)//math.ceil(len(lens)/BATCH_SIZE):,} tok/batch")
print(f"  BFD            : {BATCH_SIZE*BLOCK_SIZE:,} tok/batch")

### Step 6 — The correctness machinery: block-diagonal mask + `position_ids`

Two functions, and the second one contains the trap.

- **`block_diag_causal_mask(doc_ids)`** builds a `(B, 1, L, L)` **additive** float mask: `0.0` where attention is allowed, `finfo.min` where it is not. Allowed means *same segment* **and** *causal*. Transformers passes a 4D `attention_mask` straight through to the attention implementation instead of building its own causal mask.
- **The NaN trap:** a bin's padding region belongs to no segment, so a strictly block-diagonal mask masks those rows **entirely** → `softmax(all −inf)` → `NaN` that spreads across the batch. **The diagonal is therefore always allowed**, so a pad position attends to itself, produces a finite row, and contributes nothing (its label is `-100`).
- **The mask dtype is a trap.** `torch`'s SDPA accepts `attn_mask` only as **bool**, **float32**, or **exactly the query dtype** — and the query dtype is genuinely hard to predict here: bitsandbytes' `Linear4bit` returns its *input* dtype (inherited from `config.torch_dtype`, often bf16) rather than `bnb_4bit_compute_dtype`, and `prepare_model_for_kbit_training` plus autocast move it again during training. **float32 is the only choice that is always accepted**, so that is what this builds.
- Memory is `B·L²·4 B`: **8 MB** here, **537 MB** at `B=8, L=4096`. This is the wall that forces varlen kernels on real workloads.

In [ ]:
"""
torch's SDPA accepts an attn_mask that is bool, float32, or EXACTLY the query dtype —
nothing else. And the query dtype here is not obvious: bitsandbytes' Linear4bit returns its
INPUT dtype (which comes from config.torch_dtype, often bf16), not bnb_4bit_compute_dtype,
and prepare_model_for_kbit_training + autocast shift it again during training. float32 is
the one choice that is always accepted, whatever the plumbing does.
"""

MASK_DTYPE = torch.float32
# Fill with fp16's min rather than fp32's: exp(-65504) is already exactly 0, and this value
# survives a later cast to fp16/bf16 without becoming -inf.
MASK_FILL = torch.finfo(torch.float16).min

def block_diag_causal_mask(doc_ids, dtype=MASK_DTYPE):
    """(B, L) segment ids -> (B, 1, L, L) additive mask. Allowed = same segment AND causal.

    doc_ids == -1 marks padding (belongs to no sequence).
    """
    B, L = doc_ids.shape
    same = doc_ids[:, :, None] == doc_ids[:, None, :]    # (B,L,L) same segment?
    real = doc_ids[:, :, None] >= 0    # query position is a real token?
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=doc_ids.device))
    keep = same & causal & real
    # THE NaN GUARD: always allow the diagonal. Without it, a fully-padded row is entirely
    # masked -> softmax over all -inf -> NaN that contaminates the whole batch.
    keep |= torch.eye(L, dtype=torch.bool, device=doc_ids.device).unsqueeze(0)

    mask = torch.zeros(B, 1, L, L, dtype=dtype, device=doc_ids.device)
    return mask.masked_fill_(~keep.unsqueeze(1), MASK_FILL)

# ---- Self-check on a tiny hand-verifiable case -----------------------------------
_d = torch.tensor([[0, 0, 1, 1, -1]])   # two 2-token segments then one pad position
_m = block_diag_causal_mask(_d)[0, 0]
_ok = _m == 0   # allowed positions are exactly 0.0 (additive mask)
assert _ok[1, 0] and _ok[1, 1], "within-segment causal attention must be allowed"
assert not _ok[2, 1], "segment 1 must NOT see segment 0 (this is the contamination bug)"
assert _ok[3, 2] and _ok[3, 3], "within-segment attention in the second segment"
assert not _ok[0, 1], "causality must still hold"
assert _ok[4, 4] and not _ok[4, 3], "padding row: diagonal only (the NaN guard)"
print("mask self-check passed — segment-wise causal, padding rows non-degenerate")
print("allowed-attention pattern (rows=query, cols=key):")
for r in _ok.int().tolist():
    print("   ", r)

### Step 7 — **The ablation**: what naive packing actually costs, in loss units

The centrepiece, and it needs **no training at all** — thirteen forward passes.

Take one BFD bin holding several complete sequences. Compute **per-sequence loss** five ways:

| Variant | `position_ids` reset | block-diagonal mask | what it represents |
|---|---|---|---|
| **unpacked (gold)** | n/a | n/a | each sequence alone — the ground truth |
| naive packed | ✗ | ✗ | what you get from a packer with no correctness work |
| positions only | ✓ | ✗ | half the fix |
| mask only | ✗ | ✓ | the other half |
| **correct packed** | ✓ | ✓ | should match gold |

The gold row is the reference: identical tokens, one sequence per forward, standard causal mask. **If the "correct packed" row does not reproduce it, the packing is wrong** — and the assert says so rather than letting you train on it. This also functions as a live check that your transformers version honours a 4D `attention_mask`; if it silently ignores it, `mask only` and `correct` collapse onto the naive numbers and the assert fires.

In [ ]:
@torch.no_grad()
def per_sequence_losses(block, use_positions, use_mask):
    """Mean loss for each sequence inside one packed block, under a given correctness config."""
    ids = torch.tensor([block["input_ids"]], device=model.device)
    labels = torch.tensor([block["labels"]], device=model.device)
    doc_ids = torch.tensor([block["doc_ids"]], device=model.device)

    kwargs = {}
    if use_mask:
        kwargs["attention_mask"] = block_diag_causal_mask(doc_ids)   # (1,1,L,L) additive
    else:
        kwargs["attention_mask"] = torch.ones_like(ids)   # plain causal, leaks
    if use_positions:
        kwargs["position_ids"] = torch.tensor([block["position_ids"]], device=model.device)
    # else: transformers derives 0..L-1, i.e. every sequence after the first is mis-positioned

    logits = model(input_ids=ids, **kwargs).logits[:, :-1, :].float()
    tl, td = labels[:, 1:], doc_ids[:, 1:]
    per_tok = F.cross_entropy(logits.transpose(1, 2), tl, ignore_index=-100, reduction="none")[0]

    out = {}
    for d in sorted(set(block["doc_ids"])):
        if d < 0:
            continue
        sel = (td[0] == d) & (tl[0] != -100)
        if sel.any():
            out[d] = (per_tok[sel].sum() / sel.sum()).item()
    return out

@torch.no_grad()
def unpacked_losses(rows_by_doc):
    """Gold reference: each sequence alone, no padding, standard causal mask."""
    out = {}
    for d, r in rows_by_doc.items():
        ids = torch.tensor([r["input_ids"]], device=model.device)
        labels = torch.tensor([r["labels"]], device=model.device)
        logits = model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits[:, :-1, :].float()
        pt = F.cross_entropy(logits.transpose(1, 2), labels[:, 1:], ignore_index=-100,
                             reduction="none")[0]
        sel = labels[0, 1:] != -100
        out[d] = (pt[sel].sum() / sel.sum()).item()
    return out

# Pick a bin holding several complete sequences — the interesting case.
probe = max(bfd_blocks, key=lambda b: len({d for d in b["doc_ids"] if d >= 0}))
seg_ids = sorted({d for d in probe["doc_ids"] if d >= 0})
print(f"probe bin: {len(seg_ids)} sequences packed into {BLOCK_SIZE} tokens")

# Recover the original rows for this bin, in segment order, to build the gold reference.
starts = [i for i, d in enumerate(probe["doc_ids"]) if i == 0 or d != probe["doc_ids"][i-1]]
rows_by_doc = {}
for st in starts:
    d = probe["doc_ids"][st]
    if d < 0:
        continue
    n = sum(1 for x in probe["doc_ids"] if x == d)
    rows_by_doc[d] = {"input_ids": probe["input_ids"][st:st+n], "labels": probe["labels"][st:st+n]}

gold = unpacked_losses(rows_by_doc)
variants = {
    "naive packed (pos ✗, mask ✗)": per_sequence_losses(probe, False, False),
    "positions only (pos ✓, mask ✗)": per_sequence_losses(probe, True, False),
    "mask only (pos ✗, mask ✓)": per_sequence_losses(probe, False, True),
    "CORRECT packed (pos ✓, mask ✓)": per_sequence_losses(probe, True, True),
}

print(f"\n{'variant':<32}{'mean loss':>11}{'mean |Δ| vs gold':>19}{'max |Δ|':>10}")
print("-" * 72)
print(f"{'unpacked (GOLD)':<32}{statistics.mean(gold.values()):>11.4f}{'—':>19}{'—':>10}")
for name, res in variants.items():
    common = [d for d in res if d in gold]
    deltas = [abs(res[d] - gold[d]) for d in common]
    print(f"{name:<32}{statistics.mean(res[d] for d in common):>11.4f}"
          f"{statistics.mean(deltas):>19.4f}{max(deltas):>10.4f}")

correct = variants["CORRECT packed (pos ✓, mask ✓)"]
err = statistics.mean(abs(correct[d] - gold[d]) for d in correct if d in gold)
naive = variants["naive packed   (pos ✗, mask ✗)"]
naive_err = statistics.mean(abs(naive[d] - gold[d]) for d in naive if d in gold)
print(f"\nnaive packing error : {naive_err:.4f} loss units  <- you would train on this")
print(f"corrected error     : {err:.4f} loss units  <- fp16 noise floor")
assert err < 0.05, (
    f"corrected packing still differs from unpacked by {err:.4f}. Either the 4D attention_mask "
    "is being ignored by this transformers version, or position_ids are not reaching the model."
)
print("\nVERDICT: block-diagonal mask + reset position_ids reproduce the unpacked loss.")
print("Compare the 'positions only' and 'mask only' rows — that is how the total error splits.")

### Step 8 — What you actually buy: FLOPs arithmetic, then three timed runs

**First the honest caveat, computed from this model's own config.** Attention scores cost `O(L²)`; projections and the MLP cost `O(L)`. Packing `N` short sequences into one long block leaves the linear cost unchanged and multiplies the attention-score cost by `N`. Whether that matters depends on the **crossover length** where the two are equal — and with a dense 4D mask you pay the full `L²` because `sdpa` still materialises the score matrix before masking it. A varlen kernel would not.

In [ ]:
cfg = model.config
d_model, n_layer = cfg.hidden_size, cfg.num_hidden_layers
d_ff, n_head = cfg.intermediate_size, cfg.num_attention_heads
n_kv = getattr(cfg, "num_key_value_heads", n_head)
d_head = d_model // n_head

# Per token, per layer: 2 FLOPs per multiply-accumulate over each projection's parameters.
mlp_params = 3 * d_model * d_ff   # gate, up, down
attn_params = d_model * d_model * 2 + 2 * d_model * (n_kv * d_head)  # q,o + k,v (GQA-aware)
linear_flops_per_tok = 2 * (mlp_params + attn_params)
# Attention scores + values, causal (~half the L x L matrix): ~2 * 2 * (L/2) * d_model per token.
attn_flops_per_tok = lambda L: 2 * d_model * L

crossover = linear_flops_per_tok / (2 * d_model)
print(f"model: d_model={d_model} d_ff={d_ff} layers={n_layer} heads={n_head} kv_heads={n_kv}")
print(f"linear FLOPs/token/layer        : {linear_flops_per_tok/1e6:.1f} M")
for L in (256, BLOCK_SIZE, 4096, 8192):
    a = attn_flops_per_tok(L)
    print(f"  L={L:<5} attention FLOPs/token/layer: {a/1e6:7.1f} M  "
          f"({100*a/(a+linear_flops_per_tok):5.1f}% of total)")
print(f"\nCROSSOVER: attention == linear at L ~ {crossover:,.0f} tokens.")
print(f"At L={BLOCK_SIZE} attention is a minor share, so packing is a large net win here.")
print("Past the crossover you need a varlen kernel (FA-2 cu_seqlens / FlexAttention BlockMask),")
print("which SKIPS the off-diagonal blocks instead of computing and masking them.")

Now the measurement that matters: **the same 600 examples, one epoch, three data layouts.** Same LoRA config, same LR, same per-device batch size — only the layout differs, so the difference in wall clock and step count *is* the packing win.

The padding baseline uses **`group_by_length=True`**, i.e. the strongest honest baseline rather than a strawman.

In [ ]:
class PackedCollator:
    """Stacks fixed-size packed blocks and builds the block-diagonal mask + position_ids."""

    def __init__(self, use_correctness=True):
        self.use_correctness = use_correctness

    def __call__(self, features):
        batch = {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }
        doc_ids = torch.tensor([f["doc_ids"] for f in features], dtype=torch.long)
        if self.use_correctness:
            # 4D mask is rebuilt per batch — storing L x L per row in Arrow would be absurd.
            batch["attention_mask"] = block_diag_causal_mask(doc_ids)
            batch["position_ids"] = torch.tensor([f["position_ids"] for f in features], dtype=torch.long)
        else:
            batch["attention_mask"] = torch.ones_like(batch["input_ids"])
        return batch


peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

def train_layout(name, ds, collator, group_by_length=False):
    """One epoch over `ds`. Everything except the DATA LAYOUT is held constant."""
    m = get_peft_model(
        prepare_model_for_kbit_training(
            AutoModelForCausalLM.from_pretrained(
                base_model_id, quantization_config=bnb_config,
                device_map="auto", attn_implementation="sdpa"),
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        ),
        peft_config,
    )
    m.config.use_cache = False

    args = TrainingArguments(
        output_dir=f"./packing_{name}",
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="constant",     # constant LR: we are timing, not tuning
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        group_by_length=group_by_length,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=USE_BF16, fp16=not USE_BF16,
        optim="paged_adamw_8bit",
        logging_steps=25,
        save_strategy="no",
        report_to="none",
        label_names=["labels"],
        remove_unused_columns=False,      # keeps doc_ids/position_ids alive for the collator
        seed=42,
    )
    trainer = Trainer(model=m, args=args, train_dataset=ds,
                      data_collator=collator, processing_class=tokenizer)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = trainer.train()
    wall = time.time() - t0

    res = {"name": name, "rows": len(ds), "wall_s": wall,
           "steps": out.global_step, "loss": out.training_loss,
           "peak_gb": torch.cuda.max_memory_allocated() / 1e9}
    del trainer, m
    gc.collect(); torch.cuda.empty_cache()
    return res

In [ ]:
# The unpacked baseline needs padding, so it uses the standard seq2seq collator.
pad_ds = Dataset.from_list([{"input_ids": r["input_ids"], "labels": r["labels"]} for r in rows])
pad_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=None, padding=True,
                                      label_pad_token_id=-100, pad_to_multiple_of=8)

wrapped_ds = Dataset.from_list(wrapped_blocks)
bfd_ds = Dataset.from_list(bfd_blocks)

# The Step 7 ablation is done with `model`; reclaim its VRAM before four more get built.
# (Re-running Step 7 after this cell needs a re-run of Step 1.)
del model
gc.collect()
torch.cuda.empty_cache()

timings = [
    # Two baselines on purpose: the NAIVE one people actually ship, and the STRONG one
    # (length-grouped) that packing has to beat to be worth the complexity.
    train_layout("pad_random", pad_ds, pad_collator, group_by_length=False),
    train_layout("pad_grouped", pad_ds, pad_collator, group_by_length=True),
    train_layout("wrapped", wrapped_ds, PackedCollator(use_correctness=True)),
    train_layout("bfd", bfd_ds, PackedCollator(use_correctness=True)),
]
print("\nall four layouts trained")

In [ ]:
# ---- The result: same corpus, same hyperparameters, three data layouts -----------
real = sum(lens)
base = next(t for t in timings if t["name"] == "pad_random")     # the naive baseline
_wf, _bf = len(wrapped_blocks) * BLOCK_SIZE, len(bfd_blocks) * BLOCK_SIZE
# (forwarded, real tokens actually processed) — wrapped drops its ragged tail.
fwd_by = {"pad_random": (tokens_forwarded_padded(lens, BATCH_SIZE), real),
          "pad_grouped": (tokens_forwarded_padded(lens, BATCH_SIZE, sort_by_length=True), real),
          "wrapped": (_wf, _wf),
          "bfd": (_bf, real)}

print(f"{'layout':<14}{'rows':>6}{'steps':>7}{'fwd tokens':>12}{'util':>7}{'tok/batch':>11}"
      f"{'wall s':>9}{'real tok/s':>12}{'speedup':>9}{'peak GB':>9}")
print("-" * 98)
for t in timings:
    fwd, seen = fwd_by[t["name"]]
    print(f"{t['name']:<14}{t['rows']:>6}{t['steps']:>7}{fwd:>12,}{100*seen/fwd:>6.1f}%"
          f"{fwd//max(1, t['rows']//BATCH_SIZE):>11,}"
          f"{t['wall_s']:>9.1f}{seen/t['wall_s']:>12,.0f}"
          f"{base['wall_s']/t['wall_s']:>8.2f}x{t['peak_gb']:>9.2f}")

grouped = next(t for t in timings if t["name"] == "pad_grouped")
bfd_t = next(t for t in timings if t["name"] == "bfd")
print(f"\nreal tokens in corpus: {real:,} (identical for every row — only the LAYOUT differs)")
print(f"'speedup' is versus pad_random, the naive baseline.")
print(f"vs the STRONG baseline (pad_grouped): BFD is {grouped['wall_s']/bfd_t['wall_s']:.2f}x")
print("\nWhy the measured speed-up is SMALLER than the utilisation ratio implies:")
print("  1. group_by_length already recovers MOST of padding's waste at modest batch sizes —")
print("     check its utilisation above; against that baseline packing's win is not fewer")
print("     tokens but far fuller batches (see tok/batch), i.e. better hardware utilisation")
print("     and amortised per-step overhead;")
print("  2. the dense 4D mask buys correctness, NOT sparsity — sdpa still computes all L^2")
print("     scores and then masks them. A varlen kernel (Ampere+) is what converts the")
print("     utilisation win into the full wall-clock win.")
print("Note the step count too: packing means far fewer, much fuller optimizer steps, which")
print("changes your effective batch size and therefore the LR you should be using.")

---

## **[Key Observations]**

*Packing is judged on two numbers that must be reported together: **throughput gained** and **packing error in loss units**. One without the other is meaningless — a fast, silently-corrupted run is worse than a slow correct one.*

### Corpus & configuration

| Setting | Value |
|---|---|
| Corpus · `N_EXAMPLES` | `databricks/databricks-dolly-15k` · |
| `BLOCK_SIZE` (`L`) · `BATCH_SIZE` | |
| Length: min / median / mean / p90 / max | |
| Length **stdev** | *(the variance is what packing monetises)* |
| Examples dropped (longer than `L`) | |

### Utilisation (Step 3 / Step 5, arithmetic only)

| Scheme | Forwarded tokens | Utilisation | Waste |
|---|---|---|---|
| static pad to `L` | | | |
| dynamic pad, B=1 | | | |
| dynamic pad, B=2 | | | |
| dynamic pad, B=8 | | | |
| dynamic pad, B=16 | | | |
| dynamic pad + `group_by_length`, B=8 | | | |
| wrapped | | | |
| BFD | | | |

- Does dynamic padding get monotonically **worse** as `B` grows, as predicted?
- Wrapped vs BFD utilisation gap: ___ pts. Sequences split by wrapped: ___

### Correctness ablation (Step 7) — **the row that decides whether to ship**

| Variant | `pos` | `mask` | mean loss | mean \|Δ\| vs gold | max \|Δ\| |
|---|---|---|---|---|---|
| unpacked (GOLD) | — | — | | — | — |
| naive packed | ✗ | ✗ | | | |
| positions only | ✓ | ✗ | | | |
| mask only | ✗ | ✓ | | | |
| CORRECT packed | ✓ | ✓ | | *(≈ fp16 noise)* | |

- **Which half of the fix mattered more** on this corpus — positions or the mask? Why might that flip with a different mean sequence length?
- Naive packing error: ___ loss units. Would you have noticed it from the training curve alone?

### Throughput (Step 8)

| Layout | Steps | Forwarded tokens | Wall s | Real tok/s | Speed-up | Peak GB |
|---|---|---|---|---|---|---|
| pad + `group_by_length` | | | | | 1.00× | |
| wrapped | | | | | | |
| BFD | | | | | | |

| | Value |
|---|---|
| Attention share of FLOPs at `L` = ___ | ___ % |
| Crossover `L` (attention == linear) | |
| Real tokens per optimizer step: padded → packed | ___ → ___ |
| **Did you adjust the LR for the larger effective batch?** | |

### Ablations worth the compute

- **`BLOCK_SIZE ∈ {512, 1024, 2048, 4096}`** — utilisation rises, attention share rises, 4D-mask memory rises as `L²`. Find your knee.
- **`PackedCollator(use_correctness=False)`** — train the naive way and compare final eval loss. This is what most packing code actually does.
- **A short-sequence corpus** (`tatsu-lab/alpaca`, mean ≈ 80 tok) — utilisation under static padding should be catastrophic (~8 %) and the speed-up correspondingly large.
- **An extreme-variance corpus** (`allenai/tulu-3-sft-mixture`) — where BFD's no-split guarantee should separate it clearly from wrapped.
- **On Ampere+**: `attn_implementation="flash_attention_2"` with `position_ids`-derived `cu_seqlens`, versus this dense 4D mask. Expect equal loss and materially better speed/memory.

---

## Production Wiring — the one-flag versions

Everything above is the mechanism. In production you use a library that implements it, and now you know what its flags actually do — and which of them your hardware can honour.

| Stack | Packing | Block-diagonal attention |
|---|---|---|
| **TRL** | `SFTConfig(packing=True, packing_strategy="bfd")` (also `"wrapped"`) | emits `position_ids`; needs FlashAttention-2 for true varlen |
| **Axolotl** | `sample_packing: true` | `pad_to_sequence_len` + FA-2 varlen |
| **Raw transformers** | your own packer (Steps 4–5) | 4D mask (Step 6) or FA-2 `cu_seqlens` |
| **FlexAttention** (torch ≥ 2.5) | any packer | `create_block_mask` from `doc_ids` — sparse, so it is *also* faster |

**The rule to carry away:** `packing=True` alone is a **throughput** flag. It becomes a **correctness-preserving** flag only when the attention implementation can express document boundaries — FA-2 varlen, FlexAttention, or an explicit 4D mask. On hardware that cannot (a T4), you keep correctness via the dense mask and accept that you bought utilisation, not sparsity.

In [ ]:
# The library equivalent of Steps 4-7, for reference. Requires Ampere+ for the varlen path.
print("""
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    packing=True,                       # Step 4/5: fill the context window
    packing_strategy="bfd",             # "bfd" = Step 5 (no splitting) | "wrapped" = Step 4
    max_length=1024,                    # L
    completion_only_loss=True,          # keep the -100 prompt mask through packing
    model_init_kwargs={"attn_implementation": "flash_attention_2"},   # Ampere+ ONLY
)
# With flash_attention_2, TRL's position_ids become cu_seqlens inside the kernel: the
# off-diagonal blocks are never materialised -> correct AND faster.
# On a T4 (sm_75) FA-2 is unavailable, so use the dense 4D mask from Step 6 instead.
""")

print(f"this runtime: sm_{major}{_minor} -> "
      + ("FA-2 varlen available: use the TRL config above."
         if HAS_FA2 else
         "FA-2 NOT available (Turing). Keep the Step 6 dense mask; you get utilisation, not sparsity."))

## Export — the packing functions & measurements (Optional)

The deliverable of this notebook is not a model — it is the packers and the numbers. Both are worth keeping.

In [ ]:
import shutil, os, json

os.makedirs("./packing_artifacts", exist_ok=True)
summary = {
    "block_size": BLOCK_SIZE,
    "n_examples": len(rows),
    "real_tokens": real,
    "length_stats": {"median": statistics.median(lens), "mean": statistics.mean(lens),
                     "stdev": statistics.stdev(lens), "max": max(lens)},
    "utilisation": {k: round(100 * seen / fwd, 2) for k, (fwd, seen) in fwd_by.items()},
    "blocks": {"wrapped": len(wrapped_blocks), "bfd": len(bfd_blocks)},
    "packing_error_loss_units": {"naive": round(naive_err, 5), "corrected": round(err, 5)},
    "timings": timings,
}
with open("./packing_artifacts/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

output_filename = "packing_artifacts.zip"
shutil.make_archive(output_filename.replace(".zip", ""), "zip", "./packing_artifacts")
print(json.dumps(summary, indent=2)[:900])
print(f"\nFile: {output_filename} ({os.path.getsize(output_filename)/1e3:.1f} KB)")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did the measurement cells finish?")